# OpenEnv Wordle con GRPO usando TRL

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huggingface/trl/blob/main/examples/notebooks/openenv_wordle_grpo.ipynb)

![trl banner](https://huggingface.co/datasets/trl-lib/documentation-images/resolve/main/trl_banner_dark.png)


Con [**Transformers Reinforcement Learning (TRL)**](https://github.com/huggingface/trl), puedes entrenar un modelo que aprende a **jugar al Wordle**, un juego de adivinanza de palabras, mediante interacción y refuerzo.

- [Repositorio TRL en GitHub](https://github.com/huggingface/trl) — ¡danos una estrella para apoyar el proyecto!
- [Ejemplos Oficiales de TRL](https://huggingface.co/docs/trl/example_overview)
- [Tutoriales de la Comunidad](https://huggingface.co/docs/trl/community_tutorials)
- [OpenEnv](https://github.com/meta-pytorch/OpenEnv)


Un **entorno agente** es un contexto en el que un modelo puede tomar acciones, observar resultados y ajustar su comportamiento basándose en la retroalimentación, de forma similar a cómo los humanos aprenden por ensayo y error.
En este caso, el agente interactúa con el entorno **Wordle** a través del framework [**OpenEnv**](https://github.com/meta-pytorch/OpenEnv), que estandariza entornos de texto multi-agente y de estilo RL.

[Wordle](https://en.wikipedia.org/wiki/Wordle) es un popular puzzle de palabras en el que el jugador debe adivinar una palabra secreta de cinco letras en seis intentos.
Tras cada intento, la retroalimentación indica si cada letra es:
- **VERDE (G)**: Correcta y en la posición correcta
- **AMARILLA (Y)**: Presente pero en la posición incorrecta
- **GRIS (X)**: No está en la palabra

Este bucle de retroalimentación hace del Wordle un entorno perfecto para **RL con LLMs**, donde el objetivo es maximizar la probabilidad de adivinar la palabra correcta de forma eficiente.


Ajustaremos un modelo usando **GRPO** (Optimización de Política Relativa de Grupo) mediante TRL.
Usando `environment_factory`, el trainer gestiona automáticamente:
1. La creación de instancias del entorno para cada rollout.
2. La generación de completaciones del modelo y el análisis de llamadas a herramientas.
3. El avance por el entorno con las acciones del modelo.
4. La recolección de recompensas y la gestión del bucle de interacción.

Esto significa que solo necesitas definir la clase del entorno y la función de recompensa — el trainer se encarga del resto.
### Instalar dependendcias 
Empezamos instalando TRL (soportado con vLLM), el entorno de OpenEnv Wordle y trackio para iniciar sesión.


In [ ]:
!pip install -Uq trl[vllm] git+https://huggingface.co/spaces/openenv/wordle trackio

### Iniciar sesión en Hugging Face

Inicia sesión en tu cuenta de **Hugging Face** para guardar tu modelo ajustado, hacer seguimiento de los resultados de tus experimentos directamente en el Hub o acceder a modelos restringidos. Puedes encontrar tu **token de acceso** en la [página de configuración de tu cuenta](https://huggingface.co/settings/tokens).


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Definir el prompt de sistema

Este prompt instruye al modelo sobre cómo jugar al Wordle. Incluye las reglas del juego, el formato de retroalimentación y, lo que es importante, indica al modelo que use la herramienta `guess` para enviar sus intentos. El patrón `environment_factory` usa llamadas a herramientas para interactuar con el entorno, por lo que el modelo necesita saber qué herramienta llamar.


In [ ]:
prompt = """You are an expert Wordle solver with deep knowledge of English vocabulary, letter frequency patterns, and optimal guessing strategies.

Follow these rules to play Wordle:

1. The target is a 5-letter English word
2. You have 6 attempts to guess the correct word
3. After each guess, you receive color-coded feedback:
   - GREEN (G): Letter is correct and in the correct position
   - YELLOW (Y): Letter is in the word but in the wrong position
   - GRAY (X): Letter is not in the word at all
4. All guesses must be valid 5-letter English words
5. You cannot reuse a word you've already guessed
6. Use the tool `guess` to make a guess.
"""

## Definir el entorno

La clase `WordleEnv` envuelve el entorno OpenEnv TextArena Wordle en la interfaz esperada por `environment_factory`.

Cuando pasas `environment_factory=WordleEnv` al trainer, este:
1. Crea una nueva instancia de `WordleEnv()` para cada episodio de rollout.
2. Llama a `reset()` para iniciar una nueva partida (devuelve la observación inicial o `None`).
3. Genera automáticamente completaciones del modelo, analiza las llamadas a herramientas e invoca los métodos correspondientes (p. ej., `guess(...)`).
4. Repite hasta que el entorno señala `done=True` o se alcanza la longitud máxima de completación.

El entorno expone sus métodos públicos como herramientas. Cualquier método público (distinto de `reset`) con docstring se descubre automáticamente y se expone como herramienta invocable. Aquí, el método `guess` permite al modelo enviar un intento de Wordle y recibir retroalimentación.

Para este ejemplo, nos conectamos al entorno alojado en [openenv/wordle](https://huggingface.co/spaces/openenv/wordle).
Para uso en producción, recomendamos duplicar el Space en tu propia cuenta o ejecutarlo localmente mediante Docker, ya que las versiones alojadas tienen concurrencia limitada.

Para más información, consulta la [documentación TRL-OpenEnv](https://huggingface.co/docs/trl/main/en/openenv).


In [ ]:
from textarena_env import TextArenaAction, TextArenaEnv


class WordleEnv:
    def __init__(self):
        self.client = TextArenaEnv(base_url="https://openenv-wordle.hf.space")

    def reset(self, **kwargs) -> None | str:
        result = self.client.reset()
        # The game returns cumulative feedback each turn (new text appended at the end), so
        # we store the previous full response and slice out only the newly appended part.
        self._last_full_feedback = result.observation.messages[0].content
        self.reward = 0.0
        self.done = False
        return self._last_full_feedback

    def guess(self, guess: str) -> str:
        """
        Make a guess in the Wordle environment.

        Args:
            guess: The guessed word, formatted as '[abcde]'

        Returns:
            The feedback message from the environment.
        """
        if self.done:
            raise ValueError("Game over.")
        result = self.client.step(TextArenaAction(message=guess))
        _full_feedback = result.observation.messages[0].content
        # Just take the new feedback since the last guess
        feedback = _full_feedback[len(self._last_full_feedback):]
        self._last_full_feedback = _full_feedback
        # Penalize invalid moves
        if "You attempted an invalid move" in feedback:
            self.reward = 0.0
        else:
            self.reward = result.reward
        self.done = result.done
        return feedback

## Definir la función de recompensa

La función de recompensa recibe la lista de instancias del entorno al finalizar cada episodio. Dado que `WordleEnv` registra su propia recompensa (actualizada tras cada llamada a `guess`), simplemente la leemos.

Esto es mucho más sencillo que definir múltiples funciones de recompensa manualmente — el entorno ya conoce el resultado de la partida.


In [ ]:
def reward_func(environments, **kwargs) -> list[float]:
    return [env.reward for env in environments]

## Crear el dataset

Creamos un dataset con prompts repetidos para controlar el número de episodios de entrenamiento.
Cada entrada activa un episodio de rollout durante el entrenamiento. El prompt se formatea como un mensaje de chat.


In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({"prompt": [[{"role": "user", "content": prompt}] for _ in range(3000)]})

## Configurar GRPO

A continuación, definimos el **GRPOConfig**, que controla todos los parámetros clave del entrenamiento.
Esta configuración especifica cómo el modelo interactúa con vLLM, gestiona la memoria y registra los resultados.

Observa el parámetro `chat_template_kwargs={"enable_thinking": False}` — este deshabilita el modo de pensamiento de Qwen3 para que el modelo responda directamente con llamadas a herramientas en lugar de generar primero tokens de razonamiento interno.


In [ ]:
from trl import GRPOConfig

model_name = "Qwen/Qwen3-1.7B"
output_dir = "wordle-grpo-Qwen3-1.7B"

grpo_config = GRPOConfig(
    # Training schedule / optimization
    num_train_epochs=1,
    learning_rate=1e-6,
    gradient_accumulation_steps=64,
    per_device_train_batch_size=1,
    warmup_steps=10,
    optim="adamw_torch",
    max_grad_norm=1.0,

    # GRPO configuration
    num_generations=2,
    max_completion_length=1024,
    log_completions=True,
    num_completions_to_print=2,
    chat_template_kwargs={"enable_thinking": False},

    # vLLM configuration
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.15,
    vllm_max_model_length=3072,

    # Logging / reporting
    output_dir=output_dir,
    report_to="trackio",
    trackio_space_id=output_dir,
    logging_steps=1,
    save_steps=10,
    save_total_limit=1,

    # Memory optimization
    gradient_checkpointing=True,

    # Hub integration
    push_to_hub=True,
)

## Crear el `GRPOTrainer` e iniciar el entrenamiento

Ahora inicializamos el `GRPOTrainer` con `environment_factory=WordleEnv`.

Esto indica al trainer que gestione automáticamente todo el bucle de interacción:
- Crea una instancia de `WordleEnv` para cada episodio.
- Genera completaciones del modelo, analiza las llamadas a herramientas (como `guess`) y avanza por el entorno.
- Recolecta recompensas y gestiona el `tool_mask` (qué tokens son generados por el modelo frente a los generados por el entorno) de forma automática.

No es necesario escribir un `rollout_func` personalizado ni gestionar la tokenización manualmente.


In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model_name,
    reward_funcs=reward_func,
    train_dataset=dataset,
    args=grpo_config,
    environment_factory=WordleEnv,
)

Mostrar estadísticas de memoria antes del entrenamiento


In [ ]:
import torch

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

¡Y a entrenar!


In [ ]:
trainer_stats = trainer.train()

Mostrar estadísticas de memoria después del entrenamiento


In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_training = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
training_memory_percentage = round(used_memory_for_training / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_training} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {training_memory_percentage} %.")

## Guardar y subir al Hub


In [ ]:
trainer.save_model(output_dir)
trainer.push_to_hub()

## Cargar el modelo ajustado y ejecutar inferencia

Ahora vamos a probar nuestro modelo ajustado cargándolo y jugando una partida de Wordle.
Usamos la misma clase `WordleEnv` para interactuar con el entorno y generamos respuestas del modelo con la inferencia estándar de Transformers.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "sergiopaniego/wordle-grpo-Qwen3-1.7B"  # Replace with your HF username or organization

fine_tuned_model = AutoModelForCausalLM.from_pretrained(model_name, dtype="float32", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
import json


def play_wordle(model, tokenizer):
    env = WordleEnv()
    initial_observation = env.reset()

    print("Initial observation:")
    print(initial_observation)
    print()

    messages = [{"role": "user", "content": prompt}]
    if initial_observation:
        messages.append({"role": "user", "content": initial_observation})

    for turn in range(6):
        if env.done:
            break

        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=False,
        )
        model_inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(**model_inputs, max_new_tokens=512)
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)

        print(f"Turn {turn + 1} - Model output: {generated_text}")

        # Try to parse tool call from the generated text
        try:
            # Try to extract a guess from tool call format or bracket format
            if "guess" in generated_text:
                # Parse JSON tool call
                start = generated_text.index("{")
                end = generated_text.rindex("}") + 1
                args = json.loads(generated_text[start:end])
                if "arguments" in args:
                    args = args["arguments"]
                guess_word = args.get("guess", "")
            else:
                # Fallback: extract from brackets [word]
                import re
                match = re.search(r"\[([a-zA-Z]{5})\]", generated_text)
                guess_word = match.group(1) if match else generated_text.strip()[:5]

            feedback = env.guess(f"[{guess_word}]")
            print(f"         Guess: {guess_word} | Reward: {env.reward}")
            print(f"         Feedback: {feedback.strip()}")
            print()

            messages.append({"role": "assistant", "content": generated_text})
            messages.append({"role": "user", "content": feedback})
        except Exception as e:
            print(f"         Error: {e}")
            break

    print(f"Game finished! Final reward: {env.reward}")
    print(f"Done: {env.done}")

¡Vamos a jugar!


In [ ]:
play_wordle(fine_tuned_model, tokenizer)